# Appendix A: Reading API Documentation (Swagger)
**Time: 30-45 minutes**

In Notebook 4 you learned *what* to look for in API documentation: the base URL, the endpoints, the parameters, the authentication. At work, that documentation usually isn't a hand-written website — most company APIs publish an auto-generated, interactive page called **Swagger UI**. It always has the same layout, so once you can read one Swagger page, you can read them all. This appendix teaches you that layout and how to translate it into `requests` calls — a skill you'll use in your first week of touching any company API.

This appendix sits outside the Data Detective storyline — no clue to collect, just the skill.

### What you'll learn
- Recognize the parts of a Swagger page: endpoints, verbs, parameters, schemas
- Fetch and inspect the machine-readable spec that generates the page
- Translate a documented endpoint into a `requests` call — query *and* path parameters
- Read a response schema so you know the data's shape before you call

In [ ]:
import requests

---
## A.1 The page — and the spec behind it

A Swagger page is generated from an **OpenAPI specification**: a JSON file that describes every endpoint, parameter, and response of an API. The page is for humans; the spec is for machines. Same information, two forms.

Open this in your browser and keep it next to the notebook: **https://petstore.swagger.io** — the "Petstore", the canonical demo API that the Swagger tools themselves use. What you're looking at:

- **Title and version** at the top, plus the server URL the API lives at
- **Endpoints grouped by tag** (here: `pet`, `store`, `user`) — each row is one endpoint
- **Color-coded verbs**: GET (blue), POST (green), PUT (orange), DELETE (red)
- **A padlock icon** on endpoints that require authentication

Because the spec is just JSON served over HTTP, you can fetch it like any API response:

In [ ]:
spec_url = "https://petstore.swagger.io/v2/swagger.json"
spec = requests.get(spec_url, timeout=10).json()

print(f"API title:   {spec['info']['title']}")
print(f"Version:     {spec['info']['version']}")
print(f"Server:      https://{spec['host']}{spec['basePath']}")
print(f"Endpoints:   {len(spec['paths'])} paths documented")

**Exercise.** Pull two things out of the spec yourself. The page in your browser shows them too — this is about confirming that the spec dict holds the same information.

Required variables:
- `api_title` — a **string**: the API's title, taken from the spec
- `api_paths` — a **list** of strings: every endpoint path in the spec

Hint: the demo above already touched both places in the spec. `list(some_dict)` turns a dict's keys into a list.

In [ ]:
spec = requests.get("https://petstore.swagger.io/v2/swagger.json", timeout=10).json()

api_title = spec["info"]["title"]
api_paths = list(spec["paths"])

print(f"api_title = {api_title}")
print(f"api_paths = {api_paths}")

---
## A.2 From a docs entry to a Python call

Click any endpoint row on the Swagger page to expand it. The **Parameters** table is the part you'll translate into code. Each parameter has:

| Column | Tells you |
|--------|-----------|
| Name | The key you'll use in Python |
| **In** | *Where* it goes: `query` → into `params={...}`, `path` → into the URL itself |
| Type | string, integer, ... |
| Required | Whether you can leave it out |

Expand **GET /pet/findByStatus** in your browser: one parameter, `status`, *in: query*, required. So the translation is `params={"status": ...}`:

In [ ]:
base_url = "https://petstore.swagger.io/v2"

response = requests.get(
    f"{base_url}/pet/findByStatus",
    params={"status": "sold"},  # the docs say: name "status", in "query", required
    timeout=10,
)
pets = response.json()
print(f"Status: {response.status_code}")
print(f"Sold pets found: {len(pets)}")
print(f"First one: {pets[0]['name']}")

**Exercise.** Path parameters now. Find the endpoint on the Swagger page that fetches **one single pet by its ID** — note that its parameter is *in: path*, so it goes into the URL, not into `params=`. Then use it: take the `id` of any pet from a `findByStatus` call, and fetch that one pet.

Required variables:
- `single_pet_endpoint` — a **string**: the path template exactly as the docs show it, curly braces included
- `pet_lookup_status` — an **integer**: the status code of your single-pet call

One honest warning: the Petstore is a shared sandbox that anyone can write to, so pets get deleted constantly. If your lookup returns **404**, that's not your bug — the pet vanished between your two calls. The checkpoint accepts both 200 and 404.

In [ ]:
single_pet_endpoint = "/pet/{petId}"

base_url = "https://petstore.swagger.io/v2"
pets = requests.get(
    f"{base_url}/pet/findByStatus",
    params={"status": "available"},
    timeout=10,
).json()
pet_id = pets[0]["id"]

pet_response = requests.get(f"{base_url}/pet/{pet_id}", timeout=10)
pet_lookup_status = pet_response.status_code

print(f"single_pet_endpoint = {single_pet_endpoint}")
print(f"pet_lookup_status   = {pet_lookup_status}")

---
## A.3 Response schemas — the shape before the call

Expand an endpoint's **Responses** section, or scroll to the **Models** list at the bottom of the page: that's the documented *shape* of the JSON you'll get back — every field, its type, and what nests inside what. This is how you know `pet["category"]["name"]` exists **before** you make a single call.

The spec holds the same models as JSON. This API uses OpenAPI version 2 (also called "Swagger 2.0"), where models live under `definitions`; on newer company APIs you'll often see them under `components` → `schemas` instead — same idea, different key.

In [ ]:
pet_model = spec["definitions"]["Pet"]

print("A Pet response has these fields:")
for field, details in pet_model["properties"].items():
    print(f"  {field:<10} ({details.get('type', 'object')})")

**Exercise.** The `store` endpoints deal in orders, not pets. Find the **Order** model and collect its field names.

Required variable:
- `order_fields` — a **list** of strings: the Order model's property names

Hint: the Order model lives in the same part of the spec as Pet did.

In [ ]:
order_fields = list(spec["definitions"]["Order"]["properties"])

print(f"order_fields = {order_fields}")

---
## A.4 Auth and "Try it out" (reference — nothing to code)

Two more parts of every Swagger page, for when you meet one at work:

**The padlock and the Authorize button.** Endpoints with a padlock require authentication. Click **Authorize** at the top of the page, paste the key the API's owning team gave you, and the page sends it with every call you make from the browser. The spec also tells you *how* the key travels — as a query parameter or in a header — which is exactly the choice you met in Notebook 4 §4.5. Forget the key and you'll get the 401/403 you triggered on purpose in Notebook 6.

**The "Try it out" button.** Inside any expanded endpoint, *Try it out* lets you fill in parameters and execute the call straight from the browser — it even shows you the exact URL it built. That's the fastest first test of a company API: confirm the endpoint does what you think *before* writing any Python. When the browser call works and your Python call doesn't, compare the URLs — the difference is your bug.

In [ ]:
# Run this cell to check your work.

required_vars = ["api_title", "api_paths", "single_pet_endpoint", "pet_lookup_status", "order_fields"]

missing = [v for v in required_vars if v not in globals() or globals()[v] is None]
assert not missing, f"Missing or unfinished: {missing}. Go back and complete those exercises."

# A.1 — the spec
assert isinstance(api_title, str), f"api_title should be a string, got {type(api_title).__name__}"
assert api_title == "Swagger Petstore", f"api_title should be 'Swagger Petstore', got {api_title!r}"
assert isinstance(api_paths, list), f"api_paths should be a list, got {type(api_paths).__name__}"
assert len(api_paths) == 14, (
    f"api_paths should hold all 14 documented paths, got {len(api_paths)}. "
    "Did you take every key, not just some?"
)
assert "/pet/findByStatus" in api_paths, "api_paths should contain '/pet/findByStatus' — check what you collected."

# A.2 — path parameters
assert single_pet_endpoint == "/pet/{petId}", (
    f"single_pet_endpoint should be the docs' path template '/pet/{{petId}}', got {single_pet_endpoint!r}"
)
assert pet_lookup_status in (200, 404), (
    f"pet_lookup_status should be 200 (found) or 404 (the sandbox pet vanished), got {pet_lookup_status}. "
    "Anything else means the URL wasn't built right — did the id end up in the path?"
)

# A.3 — schemas
assert isinstance(order_fields, list), f"order_fields should be a list, got {type(order_fields).__name__}"
assert set(order_fields) == {"id", "petId", "quantity", "shipDate", "status", "complete"}, (
    f"order_fields doesn't match the Order model, got {sorted(order_fields)}. "
    "Check the Models section at the bottom of the Swagger page."
)

print("All checks passed — you can read a Swagger page.")

---
## Where you'll use this

Every company API you meet will hand you a page that looks exactly like the Petstore. The reading order is always the same: find the endpoint in the tag groups → read its parameters table (query or path?) → check the response model → *Try it out* in the browser → translate to `requests` in Python.

Next step: ask your team which APIs they use, open the Swagger page of one, and walk that exact route. For more public practice APIs, see the "Public APIs for practice" section of the cheat sheet.